# EDA — Credit Card Fraud Detection

**Minggu 1** (ROADMAP §4). Tujuan: pahami data sebelum modeling.

Dataset: Kaggle `mlg-ulb/creditcardfraud` → `data/creditcard.csv`

## Checklist
- [ ] Cek bentuk data: 284.807 baris, kolom `Time`, `V1–V28`, `Amount`, `Class`
- [ ] Hitung rasio fraud (`Class==1`) → ~0.17% (sangat imbalanced)
- [ ] Distribusi `Amount`, `Time`, perbedaan fraud vs normal
- [ ] Fitur `V` mana yang paling membedakan fraud
- [ ] Catat insight

**DoD Minggu 1:** notebook jalan tanpa error + bisa jelaskan kenapa accuracy ≠ metrik yang benar.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Tampilan rapi
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda x: f"{x:.4f}")

print("pandas", pd.__version__, "| numpy", np.__version__)

## 1. Muat data & kenali bentuknya

In [ ]:
df = pd.read_csv("../data/creditcard.csv")
print(f"Bentuk data: {df.shape[0]:,} baris × {df.shape[1]} kolom")
df.head()

In [ ]:
# Tipe data & memori
df.info()

In [ ]:
# Statistik ringkas semua kolom
df.describe()

## 2. Kualitas data: missing & duplikat

In [ ]:
missing = df.isna().sum()
print("Total nilai kosong:", int(missing.sum()))

n_dup = df.duplicated().sum()
print(f"Baris duplikat: {n_dup:,}")
# Catatan: dataset ini umumnya bersih (tanpa NaN). Duplikat boleh dibuang saat training.

## 3. Imbalance — jantung masalah

Fraud sangat langka. Ini menentukan SELURUH strategi: model unsupervised + metrik yang benar.

In [ ]:
jumlah = df["Class"].value_counts().sort_index()
persen = df["Class"].value_counts(normalize=True).sort_index() * 100

ringkas = pd.DataFrame({"jumlah": jumlah, "persen (%)": persen})
ringkas.index = ["Normal (0)", "Fraud (1)"]
print(ringkas)
print(f"\nRasio fraud: {persen[1]:.3f}%  →  1 fraud per ~{int(jumlah[0] / jumlah[1])} transaksi normal")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))

# Skala linear: fraud nyaris tak terlihat
sns.countplot(x="Class", data=df, ax=ax[0], hue="Class", legend=False, palette=["#4C9F70", "#D7263D"])
ax[0].set_title("Skala linear — fraud nyaris tak terlihat")
ax[0].set_xticklabels(["Normal", "Fraud"])

# Skala log: baru kelihatan timpangnya
sns.countplot(x="Class", data=df, ax=ax[1], hue="Class", legend=False, palette=["#4C9F70", "#D7263D"])
ax[1].set_yscale("log")
ax[1].set_title("Skala log — ketimpangan jelas")
ax[1].set_xticklabels(["Normal", "Fraud"])

plt.tight_layout()
plt.show()

### ⚠️ Kenapa accuracy MENYESATKAN (inti DoD)

Bayangkan model paling bodoh: **selalu menebak "normal"** tanpa belajar apa pun.
Mari buktikan akurasinya secara angka.

In [ ]:
from sklearn.metrics import accuracy_score, recall_score, precision_score

y_true = df["Class"]
y_bodoh = np.zeros(len(df), dtype=int)  # model 'selalu bilang normal'

acc = accuracy_score(y_true, y_bodoh)
rec = recall_score(y_true, y_bodoh, zero_division=0)      # berapa fraud yang ketangkap
pre = precision_score(y_true, y_bodoh, zero_division=0)

print(f"Akurasi model 'selalu normal' : {acc:.4%}")
print(f"Recall (fraud ketangkap)      : {rec:.4%}")
print(f"Precision                     : {pre:.4%}")
print("\nKesimpulan: akurasi ~99.8% terlihat hebat, TAPI recall 0% —")
print("model ini tidak menangkap SATU fraud pun. Maka accuracy menipu.")
print("\u2192 Pakai Precision / Recall / F1 / PR-AUC.")

## 4. Analisis `Amount` (nominal transaksi)

In [ ]:
# Statistik Amount per kelas
df.groupby("Class")["Amount"].describe()

In [ ]:
normal = df[df["Class"] == 0]["Amount"]
fraud = df[df["Class"] == 1]["Amount"]

fig, ax = plt.subplots(1, 2, figsize=(12, 4))

# Histogram (skala log-y karena ekor panjang)
ax[0].hist(normal, bins=50, alpha=0.6, label="Normal", color="#4C9F70")
ax[0].hist(fraud, bins=50, alpha=0.6, label="Fraud", color="#D7263D")
ax[0].set_yscale("log")
ax[0].set_xlabel("Amount")
ax[0].set_ylabel("Frekuensi (log)")
ax[0].set_title("Distribusi Amount")
ax[0].legend()

# Boxplot dipotong di 500 biar pola tengah kelihatan
sns.boxplot(x="Class", y="Amount", data=df[df["Amount"] < 500], ax=ax[1],
            hue="Class", legend=False, palette=["#4C9F70", "#D7263D"])
ax[1].set_xticklabels(["Normal", "Fraud"])
ax[1].set_title("Amount < 500 per kelas")

plt.tight_layout()
plt.show()

print(f"Median Amount  — Normal: {normal.median():.2f} | Fraud: {fraud.median():.2f}")
print(f"Rata-rata      — Normal: {normal.mean():.2f} | Fraud: {fraud.mean():.2f}")

## 5. Analisis `Time`

`Time` = detik sejak transaksi pertama (data ~2 hari). Cek apakah fraud nge-cluster di waktu tertentu.

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

ax[0].hist(df[df["Class"] == 0]["Time"], bins=60, color="#4C9F70")
ax[0].set_title("Sebaran waktu — Normal")
ax[0].set_ylabel("Jumlah")

ax[1].hist(df[df["Class"] == 1]["Time"], bins=60, color="#D7263D")
ax[1].set_title("Sebaran waktu — Fraud")
ax[1].set_xlabel("Time (detik sejak transaksi pertama)")
ax[1].set_ylabel("Jumlah")

plt.tight_layout()
plt.show()

## 6. Fitur `V1`–`V28`: mana yang paling membedakan fraud?

Fitur V adalah hasil PCA (anonim). Kita urutkan berdasarkan |korelasi| dengan `Class`.

In [ ]:
korelasi = df.corr(numeric_only=True)["Class"].drop("Class")
korelasi_terurut = korelasi.reindex(korelasi.abs().sort_values(ascending=False).index)

print("10 fitur paling berkorelasi dengan fraud:")
print(korelasi_terurut.head(10))

plt.figure(figsize=(10, 5))
warna = ["#D7263D" if v < 0 else "#4C9F70" for v in korelasi_terurut.values]
korelasi_terurut.plot(kind="bar", color=warna)
plt.title("Korelasi tiap fitur dengan Class (fraud)")
plt.ylabel("Korelasi Pearson")
plt.axhline(0, color="black", lw=0.8)
plt.tight_layout()
plt.show()

In [ ]:
# Distribusi 4 fitur paling diskriminatif: fraud vs normal
top4 = korelasi_terurut.abs().sort_values(ascending=False).head(4).index.tolist()

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for col, ax in zip(top4, axes.ravel()):
    sns.kdeplot(df[df["Class"] == 0][col], ax=ax, label="Normal", fill=True, color="#4C9F70")
    sns.kdeplot(df[df["Class"] == 1][col], ax=ax, label="Fraud", fill=True, color="#D7263D")
    ax.set_title(f"Distribusi {col}")
    ax.legend()

plt.tight_layout()
plt.show()

## 7. Heatmap korelasi antar fitur

Fitur V hasil PCA → harusnya saling ortogonal (korelasi antar-V rendah). Cek visual.

In [ ]:
plt.figure(figsize=(12, 9))
sns.heatmap(df.corr(numeric_only=True), cmap="coolwarm", center=0,
            square=True, linewidths=0.2, cbar_kws={"shrink": 0.7})
plt.title("Korelasi antar seluruh fitur")
plt.tight_layout()
plt.show()

## 8. Insight & Kesimpulan (DoD Minggu 1)

> Isi/koreksi poin di bawah setelah menjalankan semua sel di atas.

1. **Bentuk data:** 284.807 transaksi, 31 kolom (`Time`, `V1–V28`, `Amount`, `Class`). Tidak ada nilai kosong.
2. **Imbalance ekstrem:** fraud hanya ~0.17% (±492 kasus). 1 fraud per ~578 transaksi normal.
3. **Kenapa accuracy salah:** model "selalu normal" mencapai akurasi ~99.8% tapi **recall 0%** — tidak menangkap satu fraud pun. → wajib **Precision / Recall / F1 / PR-AUC**.
4. **Amount:** fraud cenderung nominal kecil; ada beberapa outlier. Bukan pembeda tunggal yang kuat.
5. **Fitur diskriminatif:** beberapa fitur V (mis. V14, V12, V10, V17) paling memisahkan fraud vs normal — kandidat sinyal kuat untuk model.
6. **Strategi modeling (Minggu 2):** unsupervised anomaly detection (ECOD + Half-Space Trees), fit pada data normal, fraud hanya untuk evaluasi.

✅ **DoD tercapai bila:** notebook jalan tanpa error & poin 3 bisa kamu jelaskan dengan lancar.